# TRACE on caclium imaging Data

This notebook demonstrates the pipeline:
1. Load multi-trial data
2. Preprocessing (normalization)
3. Fit a `TRACE` estimator and extract embeddings
4. Visualize the 2D embedding

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

import trace_neuro
from trace_neuro import TRACE, knn_accuracy, ari_score
from trace_neuro.data_loaders import load_data_sc

print(f"trace_neuro version: {trace_neuro.__version__}")

## 1. Load the SC dataset

`load_data_sc` reads `chirp_trials.h5` and `bar_trials.h5`, normalises each dataset
per-neuron (z-score of the trial-averaged response), and trims the first second of the
chirp stimulus (adaptation artefact).  It returns two arrays and the cell-type labels.

| array | shape | description |
|---|---|---|
| `data_chirp` | `(n_neurons, n_trials, n_time_chirp)` | chirp responses, z-scored |
| `data_bar`   | `(n_neurons, n_trials, n_time_bar)`   | bar responses, z-scored  |

In [ ]:
DATA_PATH = "/gpfs01/berens/data/data/superior_colliculus"

data_chirp, data_bar, labels, type_names = load_data_sc(
    filepath=DATA_PATH,
    trim=True,        # cut first ~1 s of chirp (9 frames)
    flatten_bar=False # use mean across 8 directions -> shape (n, trials, 2 s)
)

print("chirp shape :", data_chirp.shape, "  (neurons, trials, time)")
print("bar   shape :", data_bar.shape,   "  (neurons, trials, time)")
print("labels shape:", labels.shape)
print("cell types  :", type_names)
print("label values:", np.unique(labels))

## 2. Inspect preprocessing

The normalisation z-scores each neuron using the mean and std of its *trial-averaged*
response, then applies the same scale to every individual trial.  A well-normalised
neuron should have a trial-mean close to zero-mean and unit variance.

In [ ]:
# Trial-averaged chirp responses: (n_neurons, n_time)
chirp_mean = data_chirp.mean(axis=1)
bar_mean   = data_bar.mean(axis=1)

print("Chirp trial-mean — grand mean: {:.4f}, grand std: {:.4f}".format(
    chirp_mean.mean(), chirp_mean.std()))
print("Bar   trial-mean — grand mean: {:.4f}, grand std: {:.4f}".format(
    bar_mean.mean(), bar_mean.std()))

# Per-neuron std of trial-averaged response should be ~1
per_neuron_std_chirp = chirp_mean.std(axis=1)
per_neuron_std_bar   = bar_mean.std(axis=1)
print("\nPer-neuron std of trial-avg chirp — median: {:.4f}".format(
    np.median(per_neuron_std_chirp)))
print("Per-neuron std of trial-avg bar   — median: {:.4f}".format(
    np.median(per_neuron_std_bar)))

In [ ]:
# Visualise a few example neurons per cell type
n_examples = 3
fig, axes = plt.subplots(len(type_names), n_examples, figsize=(10, 8), sharey='row')
fig.suptitle("Example chirp trial-averaged responses (z-scored)", fontsize=12)

for row, (type_label, type_name) in enumerate(zip(np.unique(labels), type_names)):
    idx = np.where(labels == type_label)[0]
    chosen = idx[:n_examples]
    for col, neuron_idx in enumerate(chosen):
        axes[row, col].plot(chirp_mean[neuron_idx], lw=1)
        axes[row, col].axhline(0, color='gray', lw=0.5, linestyle='--')
        if col == 0:
            axes[row, col].set_ylabel(type_name, fontsize=9)
        if row == len(type_names) - 1:
            axes[row, col].set_xlabel("Time (frames)", fontsize=8)

plt.tight_layout()
plt.show()

## 3. Fit TRACE

Pass both stimuli as a list so the backbone receives the concatenated response
`[chirp | bar]` at each forward pass.  `n_trials_pp` sets how many trials are
averaged per positive pair for each stimulus separately.

In [ ]:
trace = TRACE(
    output_dim=2,
    n_epochs=200,
    batch_size=1024,
    lr=None,            # auto-scale: 0.03 * batch_size / 256
    optimizer="sgd",
    metric="euclidean",
    backbone="mlp",
    n_trials_pp=(7, 5), # 7 trials/pair for chirp, 5 for bar
    temperature=0.5,
    augmentations=False,
    device="cuda",      # change to "cpu" if no GPU available
    random_state=42,
)

# data must be a list when using multiple stimuli
data = [data_chirp, data_bar]

Z = trace.fit_transform(data)
print("Embedding shape:", Z.shape)

## 4. Evaluate

In [ ]:
knn_acc = knn_accuracy(embedding=Z, labels=labels, n_neighbors=15, verbal=True)
ari     = ari_score(embedding=Z, true_labels=labels, verbal=True)

## 5. Visualise the embedding

In [ ]:
n_types = len(np.unique(labels))
cmap = ListedColormap(sns.husl_palette(n_types).as_hex())

fig, ax = plt.subplots(figsize=(4, 4))
sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, cmap=cmap, s=3, alpha=0.85, linewidths=0)

handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=cmap(i / (n_types - 1)), markersize=7,
               label=type_names[i])
    for i in range(n_types)
]
ax.legend(handles=handles, fontsize=8, frameon=False, loc='upper right')
ax.set_title(f"TRACE embedding  |  kNN={knn_acc:.3f}  ARI={ari:.3f}", fontsize=10)
ax.set_xlabel("dim 1")
ax.set_ylabel("dim 2")
ax.set_aspect('equal')
plt.tight_layout()
plt.show()